# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`src/Conformance/Corpus.ts`, mirrored statement for statement in Python)
and commits its snapshots under `conformance/<implementation>/`. This suite checks this implementation's files are
current, that its JSON is byte-identical to every other implementation's, and that it reads every other
implementation's JSON and YAML back to the same terms. Regenerate with `npm run conformance`. The corpus has a case for
Basic and for every dialect; each case validates in its own dialect.

In [ ]:
import * as Y from "yaml";
import { readdirSync, readFileSync, statSync } from "node:fs";

import { CASES, DIALECTS, build } from "../src/Conformance/Corpus.js";
import { render } from "../src/Conformance/write.js";
import { Expressions as E } from "@mbse/expressions";
import { Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { JSON, Plain, YAML } from "@mbse/schemas/Framework";
import { assert, equal, same } from "./support.js";

const ROOT = "../../conformance";
const OWN = "typescript5";
const OTHERS = readdirSync(ROOT).filter((n) => n !== OWN && statSync(`${ROOT}/${n}`).isDirectory()).sort();
assert(same(OTHERS, ["python3"]), OTHERS.join());

const corpus = build();
const snapshots = new Map([...corpus].map(([name, [schema, root, store]]) => [name, Plain.ToPlain(store)(schema, root)]));
const yaml11 = (t: string) => Y.parse(t, { version: "1.1", intAsBigInt: true, mapAsMap: true });

## CONF-01 · This implementation's committed files are current

In [ ]:
for (const [name, text] of render(corpus)) {
  assert(readFileSync(`${ROOT}/${OWN}/${name}`, "utf8") === text, `${name} is stale: regenerate the corpus`);
}

## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
{
  const expected = CASES.flatMap((c) => [`${c}.json`, `${c}.yaml`]).sort();
  for (const name of [OWN, ...OTHERS]) assert(same(readdirSync(`${ROOT}/${name}`).sort(), expected), name);
  for (const other of OTHERS) {
    for (const name of CASES) {
      const mine = readFileSync(`${ROOT}/${OWN}/${name}.json`);
      const theirs = readFileSync(`${ROOT}/${other}/${name}.json`);
      assert(mine.equals(theirs), `${name}.json differs from ${other}`);
    }
  }
}

## CONF-03 · Every implementation's YAML reads back to exactly this implementation's snapshot, also under YAML 1.1

In [ ]:
for (const other of [OWN, ...OTHERS]) {
  for (const name of CASES) {
    const text = readFileSync(`${ROOT}/${other}/${name}.yaml`, "utf8");
    assert(equal(YAML.loads(text), snapshots.get(name)), `${other}/${name}.yaml`);
    assert(equal(yaml11(text), snapshots.get(name)), `${other}/${name}.yaml under YAML 1.1`);
  }
}

## CONF-04 · Every implementation's JSON and YAML deserialize into this implementation's store, and validate

In [ ]:
for (const other of [OWN, ...OTHERS]) {
  for (const [name, [schema, root, store]] of corpus) {
    const dialect = DIALECTS.get(name) ?? E.DIALECT;
    for (const [suffix, decode] of [["json", JSON.FromJSON(store)], ["yaml", YAML.FromYAML(store)]] as [string, any][]) {
      const restored = decode(schema, new Uint8Array(readFileSync(`${ROOT}/${other}/${name}.${suffix}`)));
      assert(F.same(restored, root) && equal(Plain.ToPlain(store)(schema, restored), snapshots.get(name)), `${other}/${name}.${suffix}`);
      assert(dialect.validate(restored, SY.free(restored, dialect)).length === 0, `${other}/${name}.${suffix}`);
    }
  }
}